In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

In [ ]:
# Константы
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'ai-forever/ruRoberta-large'
RANDOM_STATE = 42
HYPERPARAMS = {
    'lr': 0.00005,
    'weight_decay': 0.01,
    'betas': (0.9, 0.9),
    'num_epochs': 5,
    'batch_size': 8,
    'warmup_ratio': 0.1,
    'eval_steps': 100,
    'max_grad_norm': 1.0
}

### Читаем данные

In [ ]:
# Читаем доступные категории
with open('data/categories.txt', 'r') as f:
    categories = f.readlines()
for i in range(len(categories)):
    categories[i] = categories[i].replace('\n', '')
map_categories = {}
for i in range(len(categories)):
    map_categories[categories[i]] = i
print(map_categories)

In [ ]:
# Читаем размеченные и сгенерированные данные
marked_data = pd.read_csv('data/marked_data.csv')
generated_data = pd.read_csv('data/generated_data.csv')

# Исправляем ошибки автоматической разметки
marked_data = marked_data[~marked_data['category'].isin(['бытовая техника', 'электроника', 'нет категории'])]
marked_data.loc[marked_data['category'] == 'посуда', 'category'] = 'одежда'
marked_data = marked_data.rename(columns={'text': 'review'})

# Добавляем колонку с источником данных
marked_data['source'] = ['original'] * len(marked_data)
generated_data['source'] = ['generated'] * len(generated_data)

# Соединяем данные
data = pd.concat([marked_data, generated_data], axis=0)

In [ ]:
# Распределение данных по категориям
print(data['category'].value_counts())

In [ ]:
# Распределение данных по источнику
print(data['source'].value_counts())

### Подготовка данных

In [ ]:
# Загружаем токенизатор и модель
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(categories), dtype='auto', device_map='auto')

In [ ]:
# Датасет отзывов
class ReviewDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

# Подготовка батча перед подачей в модель
def collate_fn(batch, tokenizer, map_categories, device):
    reviews = [review[0] for review in batch]
    input_ids = [tokenizer(review, add_special_tokens=True, return_tensors='pt')['input_ids'].reshape(-1) for review in reviews]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id).to(device)
    attention_mask = (input_ids != tokenizer.pad_token_id).long().to(device)
    labels = torch.tensor([map_categories[review[1]] for review in batch]).long().to(device)
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

In [ ]:
# Разбиваем данные на train и test сохраняя распределение категории и источников 
data['stratify'] = data['category'] + '_' + data['source']
train, test = train_test_split(data, test_size=0.2, random_state=RANDOM_STATE, stratify=data['stratify'])
X_train, y_train = train['review'].to_list(), train['category'].to_list()
X_test, y_test = test['review'].to_list(), test['category'].to_list()

# Создаем датасеты и даталоадеры
train_dataset = ReviewDataset(X_train, y_train)
test_dataset = ReviewDataset(X_test, y_test)
train_dataloader = DataLoader(train_dataset, batch_size=HYPERPARAMS['batch_size'], shuffle=True,
                             collate_fn=lambda x: collate_fn(x, tokenizer, map_categories, DEVICE))
test_dataloader = DataLoader(test_dataset, batch_size=HYPERPARAMS['batch_size'], shuffle=True,
                            collate_fn=lambda x: collate_fn(x, tokenizer, map_categories, DEVICE))

In [ ]:
# Инициализация оптимизатора
optimizer = torch.optim.AdamW(model.parameters(), lr=HYPERPARAMS['lr'], betas=HYPERPARAMS['betas'], weight_decay=HYPERPARAMS['weight_decay'])

# Инициализация скедулера для lr
total_steps = HYPERPARAMS['num_epochs'] * len(train_dataloader)
warmup_steps = int(total_steps * HYPERPARAMS['warmup_ratio'])
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

# Загрузка лосс функции
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
best_f1 = 0.0
for epoch in range(HYPERPARAMS['num_epochs']):

    # Тренировочный цикл
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_dataloader, total=len(train_dataloader), desc=f'Epoch {epoch + 1} training'):
        optimizer.zero_grad()

        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), HYPERPARAMS['max_grad_norm'])
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
    train_loss /= len(train_dataloader)

    # Валидационный цикл
    model.eval()
    all_preds = []
    all_labels = []
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(test_dataloader, total=len(test_dataloader), desc=f'Epoch {epoch + 1} validation'):
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']
            labels = batch['labels']

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            val_loss += loss.item() 

            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    val_loss /= len(test_dataloader)
    f1 = f1_score(all_labels, all_preds, average='weighted')

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), 'output_files/best_model.pt')

    print(f'Train Loss: {train_loss}; Eval Loss: {val_loss}; F1: {f1}')